# Mini project 3

This notebook simulates **1,000 cases for 10 bonds**, converts default times to quarterly cash flows, and sums the collateral available each quarter. Cases are sorted from worst to best by the number of defaults within five years: case 1 has the most defaults, and case 1,000 has the fewest. Change `SCENARIO_ID` to inspect any ranked case.

**Conventions:** Dollar amounts are in USD; times are in years. Each bond pays a $150,000 quarterly coupon. On default, it pays recovery of 40% of face value ($4 million) once, at the end of the quarter containing default. It pays no coupon in that quarter and nothing afterward. A default exactly on a payment date takes precedence over that payment. There is no accrued coupon. The document specifies LGD but does not specify precise recovery timing, so these are explicit modeling assumptions.

**Correlation requirement:** Every pair of uncensored default times has a target population Pearson correlation of **0.20**. We calibrate the correlation between normal scores, construct their equicorrelation matrix, and use its Cholesky factor to generate correlated normal draws. This latent correlation is calibrated to meet the target after the transformation to default times. The 1,000-case sample correlations will fluctuate around the population target; sorting rows by default count does not change those correlations.

In [1]:
import numpy as np
import pandas as pd
from scipy.special import ndtr, log_ndtr, roots_hermitenorm
from scipy.optimize import brentq
from IPython.display import display

N_SCENARIOS = 1_000
N_BONDS = 10
MATURITY_YEARS = 5
PAYMENTS_PER_YEAR = 4
FACE_VALUE = 10_000_000.0
ANNUAL_COUPON_RATE = 0.06
ANNUAL_DEFAULT_PROBABILITY = 0.04
LGD = 0.60
TARGET_DEFAULT_TIME_CORRELATION = 0.20
SEED = 6103
tranches = pd.DataFrame({
    "initial_notional_usd": [20_000_000.0, 10_000_000.0],
    "annual_coupon": [0.02, 0.04],
}, index=["A", "B"])

DT = 1 / PAYMENTS_PER_YEAR
N_QUARTERS = MATURITY_YEARS * PAYMENTS_PER_YEAR
payment_times = np.arange(1, N_QUARTERS + 1) * DT
quarter_labels = [f"Q{q}" for q in range(1, N_QUARTERS + 1)]
bond_labels = [f"Bond {b}" for b in range(1, N_BONDS + 1)]
scenario_ids = pd.Index(range(1, N_SCENARIOS + 1), name="Case")
QUARTERLY_COUPON = FACE_VALUE * ANNUAL_COUPON_RATE * DT
RECOVERY = FACE_VALUE * (1 - LGD)

display(pd.Series({
    "Scenarios": N_SCENARIOS, "Bonds": N_BONDS,
    "Quarters": N_QUARTERS, "Face value per bond ($)": FACE_VALUE,
    "Quarterly coupon per bond ($)": QUARTERLY_COUPON,
    "Recovery per default ($)": RECOVERY,
}, name="Model inputs").to_frame())
display(tranches)

,Model inputs
Scenarios,1000.0
Bonds,10.0
Quarters,20.0
Face value per bond ($),10000000.0
Quarterly coupon per bond ($),150000.0
Recovery per default ($),4000000.0


,initial_notional_usd,annual_coupon
A,20000000.0,0.02
B,10000000.0,0.04


## 1. Convert annual default probability to a hazard rate

A **hazard rate** measures how quickly default can occur, given that the bond has not already defaulted. It describes default risk continuously through time.

It differs slightly from an annual default probability:

- **Annual default probability of 4%:** a surviving bond has a 4% chance of defaulting over the next full year.
- **Hazard rate:** the instantaneous rate of default among bonds that have survived so far. It is a rate per year, not itself a one-year probability.

With a constant hazard rate $\lambda$, survival through time $t$ is $S(t)=e^{-\lambda t}$, so

$$P(\text{default within }t\text{ years})=1-e^{-\lambda t}.$$

To make the one-year default probability equal to 4%, we solve:

$$0.04=1-e^{-\lambda}
\quad\Longrightarrow\quad
\lambda=-\ln(0.96)=0.0408219945\text{ per year}.$$

The notebook uses a **hazard rate of about 4.0822% per year**, distinct from the 4% one-year probability. This lets us generate default times anywhere along the timeline while preserving the required annual default probability.

The conditional quarterly default probability is $1-e^{-\lambda/4}$, and the five-year cumulative default probability is $1-e^{-5\lambda}$. 

In [2]:
hazard_rate = -np.log1p(-ANNUAL_DEFAULT_PROBABILITY)
quarterly_default_probability = -np.expm1(-hazard_rate * DT)
five_year_default_probability = -np.expm1(-hazard_rate * MATURITY_YEARS)
assert np.isclose(-np.expm1(-hazard_rate), ANNUAL_DEFAULT_PROBABILITY)
print(f"Annual hazard rate: {hazard_rate:.10f} ({hazard_rate:.6%})")
print(f"Conditional quarterly default probability: {quarterly_default_probability:.6%}")
print(f"Five-year cumulative default probability: {five_year_default_probability:.6%}")

Annual hazard rate: 0.0408219945 (4.082199%)
Conditional quarterly default probability: 1.015360%
Five-year cumulative default probability: 18.462730%


## 2. Calibrate correlation and generate default times

The target applies to **default times**, so we find the normal-score correlation $\rho$ that produces a population default-time correlation of 0.20. We build a $10\times10$ equicorrelation matrix $R$, with ones on the diagonal and calibrated $\rho$ elsewhere, then factor it as $R=LL^T$.

For each bond, generate 1,000 independent standard-normal draws. **Moment-match each column** by subtracting its sample mean and dividing by its sample standard deviation (`ddof=1`). Each column then has sample mean 0 and sample standard deviation 1; cross-column sample correlations remain close to, but not necessarily exactly, zero.

For scenario rows, apply the Cholesky factor as $X=ZL^T$, then transform the correlated normals through the standard normal CDF and inverse exponential CDF:

$$U_{s,i}=\Phi(X_{s,i}),\qquad
\tau_{s,i}=F^{-1}(U_{s,i})=-\frac{\ln(1-U_{s,i})}{\lambda}.$$

Cases are sorted from most to fewest defaults within five years. The same seed and settings reproduce the draws, moment-matched normals, default times, and case ordering.

**Correlation calibration.** For each candidate $\rho$, we integrate the distribution of one correlated pair of normal scores to calculate the implied population correlation of their exponential default times. Let $A$ and $B$ be independent standard normals:

$$X_i=A,\qquad X_j=\rho A+\sqrt{1-\rho^2}B.$$

This evaluates a bivariate normal expectation; it does not generate scenarios or replace the 10-dimensional Cholesky simulation.

By symmetry of the standard normal, $-\ln(1-\Phi(X))=-\ln\Phi(-X)$ has the same joint distribution as $g(X)=-\ln\Phi(X)$ when the normal-score correlation is $\rho$. For a standard normal $X$, $g(X)$ is exponential with mean and variance 1. Therefore,

$$\operatorname{Corr}(\tau_i,\tau_j)=E[g(X_i)g(X_j)]-1.$$

Gauss–Hermite quadrature evaluates this expectation, and a root solver finds the $\rho$ for which the population default-time correlation is 0.20. We verify the result at a higher quadrature order. The target applies to full default times, including times beyond five years.

In [3]:
def implied_default_time_correlation(rho, quadrature_order=48):
    """Population correlation of exponential times from correlated normals."""
    if not 0 <= rho <= 1:
        raise ValueError("The normal-score correlation must lie between 0 and 1.")
    nodes, weights = roots_hermitenorm(quadrature_order)
    weights = weights / np.sqrt(2 * np.pi)
    x = nodes[:, None]
    y = rho * x + np.sqrt(1 - rho**2) * nodes[None, :]
    g_x, g_y = -log_ndtr(x), -log_ndtr(y)
    expected_product = np.sum(weights[:, None] * weights[None, :] * g_x * g_y)
    return float(expected_product - 1.0)

if not 0 <= TARGET_DEFAULT_TIME_CORRELATION <= 1:
    raise ValueError("This Gaussian-copula model supports targets from 0 to 1.")
if TARGET_DEFAULT_TIME_CORRELATION in (0, 1):
    LATENT_RHO = float(TARGET_DEFAULT_TIME_CORRELATION)
else:
    LATENT_RHO = brentq(
        lambda rho: implied_default_time_correlation(rho) - TARGET_DEFAULT_TIME_CORRELATION,
        0.0, 1.0, xtol=1e-12,
    )

verified_population_correlation = implied_default_time_correlation(LATENT_RHO, quadrature_order=80)
np.testing.assert_allclose(verified_population_correlation, TARGET_DEFAULT_TIME_CORRELATION,
                           rtol=0, atol=1e-9)
np.testing.assert_allclose(implied_default_time_correlation(0.0, 80), 0.0, rtol=0, atol=1e-9)
np.testing.assert_allclose(implied_default_time_correlation(1.0, 80), 1.0, rtol=0, atol=1e-9)
print(f"Target default-time correlation: {TARGET_DEFAULT_TIME_CORRELATION:.6f}")
print(f"Calibrated normal-score correlation: {LATENT_RHO:.10f}")
print(f"Verified population default-time correlation: {verified_population_correlation:.10f}")
print(f"For comparison, normal-score correlation 0.20 implies default-time correlation "
      f"{implied_default_time_correlation(0.20, 80):.6f}.")

Target default-time correlation: 0.200000
Calibrated normal-score correlation: 0.2332349314
Verified population default-time correlation: 0.2000000000
For comparison, normal-score correlation 0.20 implies default-time correlation 0.170302.


In [4]:
rng = np.random.default_rng(SEED)
normal_correlation = np.full((N_BONDS, N_BONDS), LATENT_RHO)
np.fill_diagonal(normal_correlation, 1.0)
cholesky_factor = np.linalg.cholesky(normal_correlation)
independent_normals = rng.standard_normal((N_SCENARIOS, N_BONDS))
independent_normals = (
    independent_normals - independent_normals.mean(axis=0)
) / independent_normals.std(axis=0, ddof=1)
latent_scores = independent_normals @ cholesky_factor.T
sample_latent_correlation = np.corrcoef(latent_scores, rowvar=False)
uniforms = ndtr(latent_scores)
uniforms = np.clip(uniforms, np.finfo(float).tiny, np.nextafter(1.0, 0.0))
default_times = -np.log1p(-uniforms) / hazard_rate

# Rank cases from most to fewest defaults; stable ties keep seeded simulation order.
unranked_default_counts = (default_times <= MATURITY_YEARS).sum(axis=1)
scenario_order = np.argsort(-unranked_default_counts, kind="stable")
original_case_ids = scenario_order + 1
default_counts = unranked_default_counts[scenario_order]
rank_ids = pd.Index(range(1, N_SCENARIOS + 1), name="Rank")
case_ranking = pd.DataFrame({
    "original_case_id": original_case_ids,
    "defaults_within_5_years": default_counts,
}, index=rank_ids)

independent_normals = independent_normals[scenario_order]
latent_scores = latent_scores[scenario_order]
uniforms = uniforms[scenario_order]
default_times = default_times[scenario_order]
scenario_ids = rank_ids
default_times_df = pd.DataFrame(default_times, index=rank_ids, columns=bond_labels)

assert default_times.shape == (N_SCENARIOS, N_BONDS)
assert np.all(np.isfinite(default_times)) and np.all(default_times > 0)
assert np.allclose(independent_normals.mean(axis=0), 0.0, atol=1e-12)
assert np.allclose(independent_normals.std(axis=0, ddof=1), 1.0, atol=1e-12)
assert case_ranking["defaults_within_5_years"].is_monotonic_decreasing
assert case_ranking["original_case_id"].is_unique
assert case_ranking["original_case_id"].between(1, N_SCENARIOS).all()
assert default_counts[0] == default_counts.max()
assert default_counts[-1] == default_counts.min()
print("Case ranking by defaults within five years (highest-default cases first):")
display(case_ranking.head(10))
print(f"Lowest-default cases (ranks {N_SCENARIOS - 4}-{N_SCENARIOS}):")
display(case_ranking.tail(5))
print("Default times in years, first five ranked cases:")
display(default_times_df.head().round(4))
print(f"Defaults in rank 1 (worst): {default_counts[0]} / {N_BONDS}")
print(f"Defaults in rank {N_SCENARIOS} (best): {default_counts[-1]} / {N_BONDS}")
print(f"Defaults within five years: {default_counts.sum():,} / {default_times.size:,}")
print(f"Average sample latent-normal correlation: "
      f"{sample_latent_correlation[np.triu_indices(N_BONDS, k=1)].mean():.4f} "
      f"(target {LATENT_RHO:.4f})")

Case ranking by defaults within five years (highest-default cases first):


,original_case_id,defaults_within_5_years
Rank,,
1,140,9
2,618,9
3,760,9
4,74,8
5,216,8
6,321,8
7,562,8
8,759,8
9,850,8


Lowest-default cases (ranks 996-1000):


,original_case_id,defaults_within_5_years
Rank,,
996,978,0
997,979,0
998,981,0
999,999,0
1000,1000,0


Default times in years, first five ranked cases:


,Bond 1,Bond 2,Bond 3,Bond 4,Bond 5,Bond 6,Bond 7,Bond 8,Bond 9,Bond 10
Rank,,,,,,,,,,
1,0.1755,0.3260,2.0744,1.5734,3.7078,0.4596,0.0488,4.3922,18.2728,0.7611
2,4.6125,1.4418,1.3127,3.1634,1.4174,0.7072,0.2324,4.0991,14.5942,0.9683
3,0.1603,1.4381,0.1528,0.6211,1.4186,6.8929,1.8563,0.6154,0.2922,3.1394
4,6.4830,2.6528,0.9545,33.4137,1.5111,0.0283,4.4992,3.4479,3.5883,0.4207
5,1.6990,9.9063,2.3841,4.8846,1.4421,0.8256,0.8919,0.9452,9.8587,0.2782


Defaults in rank 1 (worst): 9 / 10
Defaults in rank 1000 (best): 0 / 10
Defaults within five years: 1,857 / 10,000
Average sample latent-normal correlation: 0.2470 (target 0.2332)


## 3. Convert default times to quarterly bond cash flows

For payment date $t_q=q/4$, coupon $C=150{,}000$, face value $F=10{,}000{,}000$, and recovery $R=4{,}000{,}000$,

$$CF_{s,i,q}=C\,\mathbf{1}_{\{\tau_{s,i}>t_q\}}
+R\,\mathbf{1}_{\{t_{q-1}<\tau_{s,i}\leq t_q\}}
+F\,\mathbf{1}_{\{q=20,\ \tau_{s,i}>5\}}.$$

The first term pays coupons while the bond survives. The second pays recovery only in the default quarter. The third repays principal at maturity only for surviving bonds. Recovery is not discounted. The project’s 9% YTM and 1% risk-free rate are for later valuation, not these contractual cash flows.

In [5]:
def bond_cash_flow_components(times):
    """Map positive default times shaped (scenario, bond) to USD arrays.

    Each returned array has shape (scenario, bond, quarter).
    Positive infinity is allowed to represent no default in a test case.
    """
    times = np.asarray(times, dtype=float)
    if times.ndim != 2 or np.isnan(times).any() or np.any(times <= 0):
        raise ValueError("Default times must be a positive 2-D array; +inf is allowed.")
    tau = times[:, :, None]
    alive = tau > payment_times
    defaults_this_quarter = (tau > (payment_times - DT)) & (tau <= payment_times)
    coupons = QUARTERLY_COUPON * alive
    recoveries = RECOVERY * defaults_this_quarter
    principal = np.zeros_like(coupons)
    principal[:, :, -1] = FACE_VALUE * (times > MATURITY_YEARS)
    return coupons, recoveries, principal

coupon_cash_flows, recovery_cash_flows, principal_cash_flows = bond_cash_flow_components(default_times)
bond_cash_flows = coupon_cash_flows + recovery_cash_flows + principal_cash_flows
assert bond_cash_flows.shape == (N_SCENARIOS, N_BONDS, N_QUARTERS)
assert np.all(bond_cash_flows >= 0)
np.testing.assert_allclose(
    recovery_cash_flows.sum(axis=2), RECOVERY * (default_times <= MATURITY_YEARS)
)
assert np.all(bond_cash_flows[default_times[:, :, None] <= (payment_times - DT)] == 0)
print("Bond cash flow array shape:", bond_cash_flows.shape)

Bond cash flow array shape: (1000, 10, 20)


## 4. Sum the 10 bonds and inspect a case

`collateral_cash_flows[rank - 1, q]` is the sum across all bonds for a ranked case and quarter. Ranks are ordered from most to fewest defaults within five years; ties keep their seeded simulation order. Use the rank slider below to inspect any case. The tables show each bond’s default timing and quarterly cash flows, plus aggregate coupons, recoveries, principal, and collateral total. Cash-flow values are in **millions of dollars**.

In [6]:
collateral_cash_flows = bond_cash_flows.sum(axis=1)
collateral_cash_flows_df = pd.DataFrame(
    collateral_cash_flows, index=scenario_ids, columns=quarter_labels
)
assert collateral_cash_flows.shape == (N_SCENARIOS, N_QUARTERS)

import ipywidgets as widgets

def inspect_scenario(rank=1):
    if not isinstance(rank, (int, np.integer)) or not 1 <= rank <= N_SCENARIOS:
        raise ValueError(f"Choose an integer rank from 1 to {N_SCENARIOS}.")
    case_position = rank - 1
    original_case_id = int(original_case_ids[case_position])
    default_quarters = [
        int(np.searchsorted(payment_times, time, side="left") + 1)
        if time <= MATURITY_YEARS else None for time in default_times[case_position]
    ]
    details = pd.DataFrame({
        "Default time (years)": default_times[case_position],
        "Default quarter": pd.array(default_quarters, dtype="Int64"),
        "Survives maturity": default_times[case_position] > MATURITY_YEARS,
    }, index=bond_labels)
    cash = pd.DataFrame(
        bond_cash_flows[case_position].T / 1e6,
        index=quarter_labels,
        columns=bond_labels,
    )
    cash["Coupons"] = coupon_cash_flows[case_position].sum(axis=0) / 1e6
    cash["Recoveries"] = recovery_cash_flows[case_position].sum(axis=0) / 1e6
    cash["Principal"] = principal_cash_flows[case_position].sum(axis=0) / 1e6
    cash["Collateral total"] = collateral_cash_flows[case_position] / 1e6
    print(
        f"Rank {rank} (original case {original_case_id}): "
        f"{default_counts[case_position]} defaults within five years; cash flows in $ millions."
    )
    display(details.round(4))
    display(cash.round(4))

SCENARIO_ID = 1
scenario_viewer = widgets.interactive(
    inspect_scenario,
    rank=widgets.IntSlider(value=SCENARIO_ID, min=1, max=N_SCENARIOS, step=1,
                           description="Rank:", continuous_update=False),
)
display(scenario_viewer)

interactive(children=(IntSlider(value=1, continuous_update=False, description='Rank:', max=1000, min=1), Outpu…

## 5. Tranche waterfall with principal recoveries

Use the Class A then Class B priority from `cdo_risk_part1`: A has $20 million initial principal and a 2% annual coupon; B has $10 million and a 4% annual coupon. Coupon cash pays quarterly tranche interest, senior first. Interest for each quarter is calculated on the opening tranche balance; principal repayments reduce balances for the next quarter. Default-quarter recoveries are principal, allocated immediately to outstanding tranche principal in A-then-B order; surviving-bond principal is allocated the same way at maturity. Repaid tranches accrue no later coupons. Any cash left after both classes are paid goes to equity. Unpaid interest is recorded but does not carry forward.

In [7]:
# 1. Build scheduled payments using the notebook's Q1,...,Q20 labels.
class_names = ["A", "B", "E"]
scheduled_tranche_cash_flows = pd.DataFrame(
    0.0, index=quarter_labels, columns=class_names[:2]
)
tranche_quarterly_coupons = {}

for tranche_name in class_names[:2]:
    notional = tranches.loc[tranche_name, "initial_notional_usd"]
    annual_rate = tranches.loc[tranche_name, "annual_coupon"]
    quarterly_payment = notional * annual_rate / PAYMENTS_PER_YEAR

    tranche_quarterly_coupons[tranche_name] = quarterly_payment
    scheduled_tranche_cash_flows[tranche_name] = quarterly_payment

    # Assumption: full tranche principal is due only at maturity.
    scheduled_tranche_cash_flows.loc[
        quarter_labels[-1], tranche_name
    ] += notional


def apply_waterfall(collateral, schedule):
    """Return USD payments shaped (rank, quarter, class): A=0, B=1, E=2."""
    collateral = np.asarray(collateral, dtype=float)
    schedule = np.asarray(schedule, dtype=float)

    if collateral.ndim != 2:
        raise ValueError("Collateral must be a scenario-by-quarter array.")

    number_of_cases, number_of_quarters = collateral.shape
    if schedule.shape != (number_of_quarters, 2):
        raise ValueError(
            "Schedule must have one row per quarter and columns A, B."
        )

    for values in (collateral, schedule):
        if not np.isfinite(values).all() or (values < 0).any():
            raise ValueError("Cash flows must be finite and nonnegative.")

    payments = np.zeros((number_of_cases, number_of_quarters, 3))

    for case_position in range(number_of_cases):
        for quarter_position in range(number_of_quarters):
            cash_available = collateral[case_position, quarter_position]
            a_due = schedule[quarter_position, 0]
            b_due = schedule[quarter_position, 1]

            a_paid = min(cash_available, a_due)
            cash_remaining = cash_available - a_paid
            b_paid = min(cash_remaining, b_due)
            equity_paid = cash_remaining - b_paid

            payments[case_position, quarter_position] = [
                a_paid, b_paid, equity_paid
            ]

    return payments


# 2. Apply to total collateral, already in rank order.
# Coupons, recoveries, and principal are already included in this array.
schedule_values = scheduled_tranche_cash_flows.to_numpy()
tranche_cash_flows = apply_waterfall(
    collateral_cash_flows, schedule_values
)

class_cash_flows = {
    class_name: pd.DataFrame(
        tranche_cash_flows[:, :, class_position],
        index=collateral_cash_flows_df.index,
        columns=collateral_cash_flows_df.columns,
    )
    for class_position, class_name in enumerate(class_names)
}

# 3. Record shortfalls without adding them to later payments due.
# Broadcast the (20, 2) schedule across all 1,000 ranked scenarios.
tranche_shortfalls = (
    schedule_values[None, :, :] - tranche_cash_flows[:, :, :2]
)

# 4. Validate shape, conservation, payment limits, and seniority.
tolerance = 1e-8

assert tranche_cash_flows.shape == (N_SCENARIOS, N_QUARTERS, 3)
assert tranche_shortfalls.shape == (N_SCENARIOS, N_QUARTERS, 2)
assert np.isfinite(tranche_cash_flows).all()
assert (tranche_cash_flows >= 0).all()
assert (tranche_shortfalls >= -tolerance).all()

np.testing.assert_allclose(
    tranche_cash_flows.sum(axis=2),
    collateral_cash_flows,
    rtol=0,
    atol=tolerance,
)

a_unpaid = tranche_shortfalls[:, :, 0] > tolerance
b_unpaid = tranche_shortfalls[:, :, 1] > tolerance
assert np.all(tranche_cash_flows[:, :, 1][a_unpaid] == 0)
assert np.all(
    tranche_cash_flows[:, :, 2][a_unpaid | b_unpaid] == 0
)

print(
    f"Waterfall checks passed for {N_SCENARIOS:,} ranked cases "
    f"x {N_QUARTERS} quarters."
)
for class_position, class_name in enumerate(class_names[:2]):
    count = np.any(
        tranche_shortfalls[:, :, class_position] > tolerance, axis=1
    ).sum()
    print(
        f"Class {class_name}: {count} of {N_SCENARIOS} cases "
        "have a payment shortfall."
    )

display(scheduled_tranche_cash_flows)

# 5. Inspect the rank specified by SCENARIO_ID.
# This reads SCENARIO_ID; moving the existing slider does not update it.
rank = SCENARIO_ID
case_position = collateral_cash_flows_df.index.get_loc(rank)

tranche_case_view = pd.DataFrame(
    tranche_cash_flows[case_position],
    index=quarter_labels,
    columns=class_names,
)
tranche_case_view.insert(
    0, "Collateral total", collateral_cash_flows[case_position]
)

print(
    f"Rank {rank} (original case {original_case_ids[case_position]}): "
    "cash flows in $ millions."
)
display((tranche_case_view / 1e6).round(4))

Waterfall checks passed for 1,000 ranked cases x 20 quarters.
Class A: 3 of 1000 cases have a payment shortfall.
Class B: 11 of 1000 cases have a payment shortfall.


,A,B
Q1,100000.0,100000.0
Q2,100000.0,100000.0
Q3,100000.0,100000.0
Q4,100000.0,100000.0
Q5,100000.0,100000.0
Q6,100000.0,100000.0
Q7,100000.0,100000.0
Q8,100000.0,100000.0
Q9,100000.0,100000.0
Q10,100000.0,100000.0


Rank 1 (original case 140): cash flows in $ millions.


,Collateral total,A,B,E
Q1,9.20,0.10,0.10,9.00
Q2,8.90,0.10,0.10,8.70
Q3,0.90,0.10,0.10,0.70
Q4,4.75,0.10,0.10,4.55
Q5,0.75,0.10,0.10,0.55
Q6,0.75,0.10,0.10,0.55
Q7,4.60,0.10,0.10,4.40
Q8,0.60,0.10,0.10,0.40
Q9,4.45,0.10,0.10,4.25
Q10,0.45,0.10,0.10,0.25


In [8]:
import ipywidgets as widgets

def show_waterfall(rank=1):
    """Display collateral, A, B, and equity payments by quarter."""
    case_id = int(case_ranking.loc[rank, "original_case_id"])

    defaulted_names = [
        bond_labels[i]
        for i in range(N_BONDS)
        if default_times[rank - 1, i] <= MATURITY_YEARS
    ]
    defaulted_text = ", ".join(defaulted_names) or "None"

    print(f"Rank {rank} of {N_SCENARIOS} (most defaults first).")
    print(f"Original case {case_id}: defaulted bonds by maturity: {defaulted_text}.")
    print("Priority: A -> B -> E. Principal assumption: bullet at maturity.")

    table = pd.DataFrame(index=quarter_labels)
    table.index.name = "Quarter"
    table["time_years"] = payment_times
    table["C_USD"] = collateral_cash_flows_df.loc[
        rank, quarter_labels
    ].to_numpy()

    for class_name in ["A", "B", "E"]:
        table[f"{class_name}_USD"] = class_cash_flows[class_name].loc[
            rank, quarter_labels
        ].to_numpy()

    cash_columns = ["C_USD", "A_USD", "B_USD", "E_USD"]
    table.loc["Total", cash_columns] = table[cash_columns].sum()

    formats = {"time_years": "{:.2f}"}
    formats.update({column: "{:,.2f}" for column in cash_columns})
    display(table.style.format(formats, na_rep=""))


waterfall_viewer = widgets.interactive(
    show_waterfall,
    rank=widgets.IntSlider(
        value=1,
        min=1,
        max=N_SCENARIOS,
        step=1,
        description="Rank:",
        continuous_update=False,
    ),
)
display(waterfall_viewer)

interactive(children=(IntSlider(value=1, continuous_update=False, description='Rank:', max=1000, min=1), Outpu…